In [1]:
import os
import sys
import numpy as np

from SciExpeM_API.SciExpeM import SciExpeM
from SciExpeM_API.Models import *

sciexp_scripts_path = r"/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts"
sys.path.append(sciexp_scripts_path)

import build_sciexp_objects
import extract_data

my_sciexpem = SciExpeM(username='YOUR_USERNAME',
                       password='YOUR_PASSWORD', secure=False, port=8080)
LABEL_DCT = {'temperature': 'T', 'time': 't',
             'length': 'l', 'pressure': 'p', 'volume': 'V'}
CONVERT_TO_BAR = {'atm': 1.01325, 'bar': 1., 'Torr': 0.00133322, 'Pa': 1e-5}

### FilePaper
 - description*
 - reference_doi*
 - author*
 - title*
 - year*
 - volume
 - page
 - journal

In [2]:
file_paper = FilePaper(reference_doi="10.1016/j.combustflame.2014.07.009",
                       author="Yuan, Wenhao; Li, Yuyang; Dagaut, Philippe; Yang, Jiuzhong; Qi, Fei",
                       title="Investigation On The Pyrolysis And Oxidation Of Toluene Over A Wide Range Conditions. I. Flow Reactor Pyrolysis And Jet Stirred Reactor Oxidation",
                       year=2015,
                       description="Yuan, Wenhao; Li, Yuyang; Dagaut, Philippe; Yang, Jiuzhong; Qi, Fei - Combustion And Flame, 2015, (162), 3-21",                       )

### Common properties

- name
- units
- value
- source_type

### OpenSMOKE input file if available

In [3]:
datapath = os.path.join(sciexp_scripts_path, 'data')
osinputname = 'PFR_profiles_inputOS.dic'
# WARNING LIST OF SPECIES MUST HAVE THE NAMES OF THE MECH YOU ARE GOING TO SIMULATE - OTHERWISE, NEED REPLACEMENT
inputstr, extrainfo = extract_data.process_osinput(
    datapath, osinputname, profiles=True, flameinfo=True)
# LA LISTA DI OUTPUTSPECIES DEVE AVERE SPECIE NEL MECCANISMO CHE SIMULERAI

reminder if sth looks weird - ListOfProfiles must be all in the same row


In [4]:
print(extrainfo)

{'profileinfo': {'csv_profiles': ['PFR_profiles/1100.csv', 'PFR_profiles/1150.csv', 'PFR_profiles/1200.csv', 'PFR_profiles/1250.csv', 'PFR_profiles/1300.csv', 'PFR_profiles/1350.csv', 'PFR_profiles/1400.csv', 'PFR_profiles/1450.csv', 'PFR_profiles/1500.csv', 'PFR_profiles/1550.csv', 'PFR_profiles/1600.csv', 'PFR_profiles/1650.csv', 'PFR_profiles/1700.csv', 'PFR_profiles/1750.csv'], 'csv_paths': ['/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts/data/PFR_profiles/1100.csv', '/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts/data/PFR_profiles/1150.csv', '/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts/data/PFR_profiles/1200.csv', '/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/

In [5]:
sourcetype = 'reported'
commonprop = []
for name, values in extrainfo['commonprop'].items():
    if name not in ['temperature']:
        ci = CommonProperty(name=name, units=values[1], value=values[0], source_type=sourcetype)
        commonprop.append(ci)
    
# ADD OTHER COMMON PROPERTIES
#ci = CommonProperty(name='temperature', units='K', value='1120', source_type='reported')
#commonprop.append(ci)
############# DO NOT EDIT
# extract pressure
if 'pressure' in extrainfo['commonprop'].keys():
    Pval, Punit = extrainfo['commonprop']['pressure']
    P = float(Pval) * CONVERT_TO_BAR[Punit]

In [6]:
for c in commonprop:
    print(c.serialize())

{'name': 'length', 'units': 'mm', 'value': '210', 'source_type': 'reported'}
{'name': 'pressure', 'units': 'Pa', 'value': '4000', 'source_type': 'reported'}


In [7]:
csv_profiles = extrainfo['profileinfo']['csv_profiles']
csv_paths = extrainfo['profileinfo']['csv_paths']
print('reminder if sth looks weird - ListOfProfiles must be all in the same row')
print(csv_profiles)

['PFR_profiles/1100.csv', 'PFR_profiles/1150.csv', 'PFR_profiles/1200.csv', 'PFR_profiles/1250.csv', 'PFR_profiles/1300.csv', 'PFR_profiles/1350.csv', 'PFR_profiles/1400.csv', 'PFR_profiles/1450.csv', 'PFR_profiles/1500.csv', 'PFR_profiles/1550.csv', 'PFR_profiles/1600.csv', 'PFR_profiles/1650.csv', 'PFR_profiles/1700.csv', 'PFR_profiles/1750.csv']


### Initial Species

- name
- units
- value
- source_type
- configuration

In [8]:
# THIS REFERS TO THE SIMULATION
# PREMIXED IS DEFAULT AND MUST BE INDICATED UNLESS IT'S A CF FLAME
comp_unit = 'mole fraction'
srctype = 'calculated'
config = 'premixed'
##############################
################# do not edit
inspecies, fuelobjs = build_sciexp_objects.build_initial_species(
    my_sciexpem=my_sciexpem,
    molefractions=extrainfo['molefractions'],
    source_type=srctype,
    units=comp_unit,
    configuration=config,
)


Species: ['C7H8', 'AR']
Fuels: ['C7H8']
Mole Frac: ['0.05', '0.95']
config: ['premixed', 'premixed']


### Data columns

- name
- label (not comuplsory)
- species_object
- units
- data
- dg_id 
- dg_label
- source_type


In [9]:
sp_table = {
    'C5H4': 'C5H4+CYC5H4',
    'C5H3': 'LC5H3-24',
    'C9H10': 'INDANE',
    'C8H8': 'C6H5C2H3',
    'C8H6': 'C6H5C2H',
    'C8H10': 'C6H5C2H5',
    'ACENAPHTHLYNE': 'C12H8',
    'ACENAPH': 'C12H8',
    'C13H10': 'FLUORENE',
    'C13H12': 'C6H5CH2C6H5',
    'C14H14': 'C6H5C2H4C6H5',
    'C14H12': 'STILBENE',
    
    #  'C10H7CH3': 'C10H7CH3+C10H7CH3_106',
}

In [10]:
# read species profiles
datafile = open(os.path.join(
    sciexp_scripts_path, 'data', 'PFR_profile_data.txt'))
# DO NOT DELETE ZERO VALUES - REACTORS WITH PROFILES MUST ALL HAVE THE SAME X AXIS!
df_data = extract_data.readdata(datafile, delzero=False)

In [11]:
# optional: if extracted data don't correspond exactly to a profile
# profiles post-processing for digitized data - necessary to associate a profile
df_data = extract_data.align_data_to_profile_nominal_values(
    df_data,
    csv_paths,
    init_condition='init-cond1',
    expected_name='temperature',
    dg_id='dg1',
    fillna=0.0,
)

In [12]:
# process data

srctype = 'digitized'
label = 'experimental_data'
x = ['temperature', 'K']
# y = ['concentration', 'mol/cm3']
# x = ['distance', 'cm']
y = ['composition', 'mole fraction']
list_exclude_species = []
uncert_x = []
uncert_y = []
#################### DO NOT EDIT################
TINF, TSUP = extract_data.temperature_bounds(
    extrainfo,
    x=x,
    df_data=df_data,
)
print('T max and min: {} - {} K'.format(TINF, TSUP))

datacols = build_sciexp_objects.build_data_columns(
    df_data=df_data,
    my_sciexpem=my_sciexpem,
    x=x,
    y=y,
    source_type=srctype,
    label=label,
    sp_table=sp_table,
    list_exclude_species=list_exclude_species,
    uncert_x=uncert_x,
    uncert_y=uncert_y,
)
dgs = list(df_data.keys())

T max and min: 1100.0 - 1750.0 K
C7H8 [<Species (50)>]
species FULVENE not as preferred name: alternative names searched
FULVENE [<Species (38)>]
C7H7 [<Species (460)>]
CYC6H4 [<Species (16)>]
species C9H8 not as preferred name: alternative names searched
C9H8 [<Species (35)>]
CH4 [<Species (6)>]
C9H10 [<Species (139)>]
C10H8 [<Species (27)>]
C8H10 [<Species (47)>]
C6H2 [<Species (41)>]
C7H6 [<Species (462)>]
C5H5 [<Species (63)>]
C2H2 [<Species (51)>]
C4H2 [<Species (11)>]
C7H5 [<Species (463)>]
C5H6 [<Species (64)>]
species NAPHTALYNE not as preferred name: alternative names searched
NAPHTALYNE []
C8H8 [<Species (46)>]
species C18H12 not as preferred name: alternative names searched
C18H12 [<Species (32)>, <Species (387)>, <Species (388)>, <Species (389)>, <Species (494)>]
C3H4-P [<Species (93)>]
C6H6 [<Species (39)>]
C14H10 [<Species (29)>]
species STILBENE not as preferred name: alternative names searched
C14H12 [<Species (464)>]
C3H3 [<Species (54)>]
C5H3 [<Species (545)>]
C13H12 

In [13]:
#PROFILES
srctype = 'reported'

########## DO NOT EDIT ###########################
datacols += build_sciexp_objects.build_profile_data_columns(
    csv_paths=csv_paths,
    existing_data_groups=dgs,
    label_dct=LABEL_DCT,
    source_type=srctype,
)

### Assembly the experiment

In [16]:
PHI_INF = extract_data.equivalence_ratio(extrainfo['molefractions'])
PHI_SUP = PHI_INF

ValueError: oxidizer species O2 not found in molefractions

In [ ]:
PHI_INF = extract_data.equivalence_ratio(extrainfo['molefractions'])
PHI_SUP = PHI_INF

e = Experiment(reactor='flow reactor', 
               experiment_type='outlet concentration measurement', #NO CAPITAL LETTERS HERE
               file_paper=file_paper, 
               reactor_modes=['premixed'], 
               data_columns=datacols, 
               initial_species=inspecies, 
               common_properties=commonprop,
               os_input_file=inputstr,
               t_inf = TINF, t_sup = TSUP,
               p_inf = P, p_sup = P,
               phi_inf = PHI_INF, phi_sup = PHI_SUP,
               fuels_object = fuelobjs,
               comment = 'extracted data averaged to trace it back to a single datagroup corresponding to the profiles\
                missing vals set to 0.0'
               #comment = 'original data reported in yield (weight%) and converted to mole fractions'
               #comment = 'C4H2 plot unclear, order might be 1e6 or 1e8 (person who digitized assumed 1e8); bath gas varies with experimental run (Ne, He), but should not affect the results'
               )

ValueError: oxidizer species O2 not found in molefractions

In [122]:
e.serialize()

{'data_columns': [{'name': 'temperature',
   'units': 'K',
   'data': [1100,
    1150,
    1200,
    1250,
    1300,
    1350,
    1400,
    1450,
    1500,
    1550,
    1600,
    1650,
    1700,
    1750],
   'dg_id': 'dg1',
   'dg_label': 'experimental_data',
   'label': None,
   'source_type': 'digitized',
   'data_group_profile': None,
   'uncertainty_kind': None,
   'uncertainty_bound': None,
   'diz': None,
   'species_object': None,
   'uncertainty_reference': None},
  {'name': 'composition',
   'units': 'mole fraction',
   'data': [0.0,
    0.0502,
    0.0,
    0.0499,
    0.0493,
    0.0486,
    0.0474,
    0.0445,
    0.0388,
    0.0294,
    0.018,
    0.0,
    0.0,
    0.0],
   'dg_id': 'dg1',
   'dg_label': 'experimental_data',
   'label': None,
   'source_type': 'digitized',
   'data_group_profile': None,
   'uncertainty_kind': None,
   'uncertainty_bound': None,
   'diz': None,
   'species_object': [50],
   'uncertainty_reference': None},
  {'name': 'composition',
   'un

### Send Experiment

In [123]:
my_sciexpem.insertElement(e, verbose=True)
# QUANDO HAI COMPLETATO IL CARICAMENTO SU SCIEXPEM TU PUOI FARE IL DOWNLOAD DI UN JSON; 

Experiment element inserted successfully.
